# Lesson 18 — k-Nearest Neighbors and Support Vector Machines

**What this notebook does:** builds **k-nearest neighbors (kNN)** from scratch on a tiny set of support tickets — measuring distance, finding the k closest stored tickets, letting them vote, and watching the answer change as k changes — then builds the core idea of a **support vector machine (SVM)**: out of all the lines that separate two classes, find the one with the **widest safety gap**. It closes with the classic trap (unscaled features) and the classic trick (adding a feature so a straight line suddenly works).

Run the cells top to bottom.

## Part 1 — k-Nearest Neighbors: the model that is just a memory

Every model so far **learned** something: a slope (Lesson 14), a probability curve (Lesson 15), a tree of questions (Lesson 16). kNN learns **nothing**. Its "training" is: *store all the labelled examples*. All the work happens later, at prediction time: when a new ticket arrives, find the **k** stored tickets most similar to it, and let them **vote**.

Each ticket here has two features:

- `urgent_words` — how many urgency words it contains ("asap", "immediately", "right now", ...)
- `caps_words` — how many words are typed in ALL CAPS

The next cell stores our 8 labelled tickets. That list **is** the entire model.

In [ ]:
# each stored ticket: (urgent_words, caps_words, label)
stored_tickets = [
    (0, 0, "not urgent"),
    (1, 0, "not urgent"),
    (0, 1, "not urgent"),
    (1, 2, "not urgent"),
    (3, 3, "urgent"),
    (4, 2, "urgent"),
    (3, 4, "urgent"),
    (5, 3, "urgent"),
]

# "training" a kNN model = just keeping this list
print("Stored tickets (this IS the whole model):")
for ticket in stored_tickets:
    urgent_words = ticket[0]      # feature 1
    caps_words = ticket[1]        # feature 2
    label = ticket[2]             # the known answer
    print("  urgent_words =", urgent_words, "| caps_words =", caps_words, "| label =", label)

## Look at the data first

Two features means we can draw every ticket as a point. The two classes form two loose groups. We also mark the two new tickets we will classify: **A (4, 4)** sits deep inside the urgent group, and **B (2, 2)** sits in the awkward middle ground — exactly where the choice of k will start to matter.

In [ ]:
import os
import matplotlib.pyplot as plt

# folder for saved plots
os.makedirs("plots", exist_ok=True)

# split the tickets by label so each class gets its own colour
not_urgent_x = []
not_urgent_y = []
urgent_x = []
urgent_y = []
for ticket in stored_tickets:
    if ticket[2] == "urgent":
        urgent_x.append(ticket[0])
        urgent_y.append(ticket[1])
    else:
        not_urgent_x.append(ticket[0])
        not_urgent_y.append(ticket[1])

plt.figure(figsize=(6, 5))
plt.scatter(not_urgent_x, not_urgent_y, color="tab:blue", s=90, label="stored: not urgent")
plt.scatter(urgent_x, urgent_y, color="tab:red", s=90, label="stored: urgent")
plt.scatter([4], [4], color="purple", marker="*", s=250, label="new ticket A (4, 4)")
plt.scatter([2], [2], color="goldenrod", marker="*", s=250, label="new ticket B (2, 2)")
plt.xlabel("urgent words")
plt.ylabel("ALL-CAPS words")
plt.title("The stored tickets ARE the kNN model")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("plots/lesson-18-knn-neighbors.png", dpi=120, bbox_inches="tight")
plt.show()

## Distance: turning "similar" into a number

"Find the most similar tickets" needs a definition of *similar*. kNN uses the straight-line **(Euclidean) distance** between the points, computed with Pythagoras: take the difference in each feature, square each difference, add them up, take the square root.

Small distance = similar tickets. That is the whole definition.

In [ ]:
import math

def euclidean_distance(x1, y1, x2, y2):
    # difference in each feature
    dx = x1 - x2
    dy = y1 - y2
    # square both, add, square-root (Pythagoras)
    return math.sqrt(dx * dx + dy * dy)

# worked example: new ticket B (2, 2) vs stored ticket (3, 3)
d = euclidean_distance(2, 2, 3, 3)
print("difference in urgent_words:", 2 - 3)
print("difference in caps_words:  ", 2 - 3)
print("squared and added: 1 + 1 = 2")
print("distance = sqrt(2) =", round(d, 3))

## Measure the distance to every stored ticket

For new ticket B (2, 2) we compute the distance to **all 8** stored tickets. This is the real cost of kNN: it does this full scan for **every single prediction**.

In [ ]:
# the borderline new ticket
new_urgent_words = 2
new_caps_words = 2

print("New ticket B: urgent_words = 2, caps_words = 2")
print()
print("Distance to every stored ticket:")

distances = []                       # will hold (distance, label) pairs
for ticket in stored_tickets:
    urgent_words = ticket[0]
    caps_words = ticket[1]
    label = ticket[2]
    d = euclidean_distance(new_urgent_words, new_caps_words, urgent_words, caps_words)
    distances.append((d, label))     # remember the pair for the vote
    print("  to (" + str(urgent_words) + ", " + str(caps_words) + ") " + label.ljust(10) + " -> distance " + str(round(d, 3)))

## Pick the k closest and let them vote

With k = 3: find the smallest distance, set it aside, and repeat three times. Then each of the 3 nearest tickets casts one vote for its own label. Majority wins. We use an **odd** k so the vote cannot tie.

In [ ]:
k = 3

# work on a copy so we can remove tickets as we pick them
remaining = distances.copy()
nearest = []

# find the smallest distance, k times
for pick_number in range(k):
    smallest = remaining[0]              # start by assuming the first is closest
    for pair in remaining:
        if pair[0] < smallest[0]:        # found a closer one
            smallest = pair
    nearest.append(smallest)             # keep it
    remaining.remove(smallest)           # so the next round finds the next-closest
    print("pick", pick_number + 1, "-> distance", round(smallest[0], 3), "| label:", smallest[1])

# Compact alternative (same result):
# distances.sort()              # sorts the pairs by distance, smallest first
# nearest = distances[:3]       # take the first three

# each of the k nearest casts one vote for its own label
urgent_votes = 0
not_urgent_votes = 0
for pair in nearest:
    if pair[1] == "urgent":
        urgent_votes = urgent_votes + 1
    else:
        not_urgent_votes = not_urgent_votes + 1

print()
print("votes: urgent =", urgent_votes, "| not urgent =", not_urgent_votes)
if urgent_votes > not_urgent_votes:
    print("k =", k, "prediction: URGENT")
else:
    print("k =", k, "prediction: NOT URGENT")

## The same ticket, three different k values

Wrap the whole procedure in one function and ask the same borderline ticket three times: k = 1, k = 3, k = 7. Watch the answer flip back and forth — k is a real decision, not a detail. A small k trusts one neighbor (jumpy, overfits noise); a big k averages over so many tickets that it drifts toward whichever class has more stored examples nearby (underfits). This is Lesson 13's bias–variance dial again, wearing a new costume.

In [ ]:
def knn_predict(new_x, new_y, k):
    # step 1: distance to every stored ticket
    pairs = []
    for ticket in stored_tickets:
        d = euclidean_distance(new_x, new_y, ticket[0], ticket[1])
        pairs.append((d, ticket[2]))

    # step 2: pick the k smallest, one at a time
    nearest = []
    for pick_number in range(k):
        smallest = pairs[0]
        for pair in pairs:
            if pair[0] < smallest[0]:
                smallest = pair
        nearest.append(smallest)
        pairs.remove(smallest)

    # step 3: count the votes
    urgent_votes = 0
    not_urgent_votes = 0
    for pair in nearest:
        if pair[1] == "urgent":
            urgent_votes = urgent_votes + 1
        else:
            not_urgent_votes = not_urgent_votes + 1

    # step 4: majority wins
    if urgent_votes > not_urgent_votes:
        return "URGENT", urgent_votes, not_urgent_votes
    else:
        return "NOT URGENT", urgent_votes, not_urgent_votes

# the same borderline ticket, three different k values
for k in [1, 3, 7]:
    answer, urgent_votes, not_urgent_votes = knn_predict(2, 2, k)
    print("k =", k, "-> votes urgent:", urgent_votes, "| not urgent:", not_urgent_votes, "-> prediction:", answer)

print()
# sanity check: a ticket deep inside the urgent group
answer, urgent_votes, not_urgent_votes = knn_predict(4, 4, 3)
print("clear ticket (4, 4) with k = 3 -> votes urgent:", urgent_votes, "| not urgent:", not_urgent_votes, "-> prediction:", answer)

## The scaling trap

Distance treats every feature's units as equally important. Put a big-range feature — message length in words (0 to 200) — next to our small-range word counts (0 to 5), and length differences completely drown the features that carry the real signal. Two stored tickets are enough to watch it break.

In [ ]:
def distance_3d(a1, a2, a3, b1, b2, b3):
    # same Pythagoras idea, one more feature
    d1 = a1 - b1
    d2 = a2 - b2
    d3 = a3 - b3
    return math.sqrt(d1 * d1 + d2 * d2 + d3 * d3)

# two stored tickets, now with a third feature: message length in words
# ticket P: mild words, short message  -> not urgent   (1, 2, 48)
# ticket Q: strong words, long message -> urgent       (3, 3, 150)

# the new ticket: mild words like P, but long like Q -> (2, 2, 150)

d_to_P = distance_3d(2, 2, 150, 1, 2, 48)
d_to_Q = distance_3d(2, 2, 150, 3, 3, 150)
print("with raw message length included as a feature:")
print("  distance to P (not urgent):", round(d_to_P, 3))
print("  distance to Q (urgent):    ", round(d_to_Q, 3))
print("  nearest is Q -> URGENT (the length feature decided everything)")

print()
d_to_P_2d = euclidean_distance(2, 2, 1, 2)
d_to_Q_2d = euclidean_distance(2, 2, 3, 3)
print("with the two word-count features only:")
print("  distance to P (not urgent):", round(d_to_P_2d, 3))
print("  distance to Q (urgent):    ", round(d_to_Q_2d, 3))
print("  nearest is P -> NOT URGENT")

## Part 2 — Support Vector Machines: the widest street

Back to Lesson 15's idea of a **decision boundary** — a line with "urgent" on one side and "not urgent" on the other. When the data is separable, *many* lines work, and they are not equally good. The SVM's rule: pick the line with the **widest margin** — the biggest distance to the closest point of either class. A wide margin leaves breathing room for tomorrow's slightly different tickets.

Our candidate lines all have the form `urgent_words + caps_words = c` (and only values of `c` that actually separate the two groups are allowed to compete). Distance from a ticket to such a line, in plain terms: the ticket's feature **sum** is some amount away from `c`, and walking straight (perpendicular) toward the line changes the sum by √2 per step of length 1 — so **distance = |sum − c| ÷ √2**.

The margin of a line = the distance to its **closest** ticket. The best line = the widest margin.

In [ ]:
# a clean, separable set of tickets for the SVM idea
svm_tickets = [
    (0, 0, "not urgent"),
    (1, 1, "not urgent"),
    (0, 2, "not urgent"),
    (4, 4, "urgent"),
    (5, 3, "urgent"),
    (4, 2, "urgent"),
]

# candidate lines: urgent_words + caps_words = c
# (every candidate below keeps all not-urgent sums on one side and all urgent sums on the other)
candidate_c_values = [2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5]

best_c = None
best_margin = None

for c in candidate_c_values:
    # a line's margin = its distance to the CLOSEST ticket
    closest_distance = None
    for ticket in svm_tickets:
        feature_sum = ticket[0] + ticket[1]                    # where the ticket sits
        distance_to_line = abs(feature_sum - c) / math.sqrt(2)
        if closest_distance is None or distance_to_line < closest_distance:
            closest_distance = distance_to_line                # new closest ticket
    print("line: sum =", c, "-> margin (closest ticket):", round(closest_distance, 3))

    # keep the widest margin seen so far
    if best_margin is None or closest_distance > best_margin:
        best_margin = closest_distance
        best_c = c

print()
print("widest street wins: sum =", best_c, "with margin", round(best_margin, 3))

## Support vectors: the only tickets that matter

The winning line sits at sum = 4 — exactly halfway between the closest "not urgent" sum (2) and the closest "urgent" sum (6). The tickets sitting **exactly at the margin distance** are called **support vectors** — they pin the street in place. Every other ticket could be deleted and the line would not move an inch.

In [ ]:
# support vectors = the tickets sitting EXACTLY at the margin distance
print("support vectors (the tickets that pin the street in place):")
for ticket in svm_tickets:
    feature_sum = ticket[0] + ticket[1]
    distance_to_line = abs(feature_sum - best_c) / math.sqrt(2)
    if abs(distance_to_line - best_margin) < 0.000001:       # allow tiny float error
        print("  (" + str(ticket[0]) + ", " + str(ticket[1]) + ") " + ticket[2])

## Draw the street

The solid line is the decision boundary (sum = 4). The dashed lines are the street's two edges (sum = 2 and sum = 6). The circled points are the support vectors touching the edges.

In [ ]:
# split the SVM tickets by label for plotting
sv_not_x = []
sv_not_y = []
sv_urg_x = []
sv_urg_y = []
for ticket in svm_tickets:
    if ticket[2] == "urgent":
        sv_urg_x.append(ticket[0])
        sv_urg_y.append(ticket[1])
    else:
        sv_not_x.append(ticket[0])
        sv_not_y.append(ticket[1])

plt.figure(figsize=(6, 5))
plt.scatter(sv_not_x, sv_not_y, color="tab:blue", s=90, label="not urgent")
plt.scatter(sv_urg_x, sv_urg_y, color="tab:red", s=90, label="urgent")
plt.plot([0, 4], [4, 0], color="black", linewidth=2, label="decision line (sum = 4)")   # the boundary
plt.plot([0, 2], [2, 0], color="gray", linestyle="--", label="street edge (sum = 2)")   # not-urgent edge
plt.plot([0, 6], [6, 0], color="gray", linestyle="--", label="street edge (sum = 6)")   # urgent edge
plt.scatter([1, 0, 4], [1, 2, 2], s=300, facecolors="none", edgecolors="black", linewidths=2, label="support vectors")
plt.xlabel("urgent words")
plt.ylabel("ALL-CAPS words")
plt.title("SVM: the widest street between the classes")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("plots/lesson-18-svm-margin.png", dpi=120, bbox_inches="tight")
plt.show()

## When no straight line works: the kernel idea

One feature this time: a customer's star rating, centered on 3 stars (so 1★ becomes −2 and 5★ becomes +2). Tickets with **extreme** ratings (furious or thrilled) get a human follow-up; middling ones are routine. No single threshold on the raw rating can catch **both** ends — we try every threshold and count the mistakes. (Pointing the rule the other way, `rating >= t`, just mirrors the same failures.)

Then we manufacture ONE new feature, `rating × rating`: extremes become big, middles stay small — and suddenly one clean threshold separates the classes perfectly. Real SVMs use a mathematical shortcut called a **kernel** to get the effect of lifted features like this without building them by hand.

In [ ]:
# one feature: star rating centered on 3 (1 star -> -2, 5 stars -> +2)
ratings = [-2, -1, 0, 1, 2]
labels = ["follow-up", "routine", "routine", "routine", "follow-up"]

# try every straight threshold: "rating <= t -> follow-up, otherwise routine"
print("straight thresholds on the raw rating ('rating <= t -> follow-up'):")
for t in [-1.5, -0.5, 0.5, 1.5]:
    errors = 0
    for i in range(len(ratings)):
        if ratings[i] <= t:
            prediction = "follow-up"
        else:
            prediction = "routine"
        if prediction != labels[i]:
            errors = errors + 1              # count every wrong ticket
    print("  t =", t, "-> errors:", errors, "out of 5")

print()
# manufacture ONE new feature: the squared rating
squared = []
for r in ratings:
    squared.append(r * r)                    # extremes become big, middles stay small
print("squared feature (rating * rating):", squared)

print()
print("rule on the new feature: 'squared >= 2.5 -> follow-up'")
errors = 0
for i in range(len(squared)):
    if squared[i] >= 2.5:
        prediction = "follow-up"
    else:
        prediction = "routine"
    if prediction != labels[i]:
        errors = errors + 1
    print("  rating", ratings[i], "-> squared", squared[i], "-> prediction:", prediction, "| actual:", labels[i])
print("errors:", errors, "out of 5")

## Wrap-up

- **kNN**: no training at all — the stored examples are the model. All the cost lands at prediction time (a full distance scan per question). The answer is a vote among the k nearest; k is the overfitting dial; distance breaks when features live on wildly different scales.
- **SVM**: of all the lines that separate the classes, take the widest street. Only the support vectors pin it in place. Kernels handle shapes a straight line cannot.
- Both are done properly by libraries later (scikit-learn, Chapter 9) — this notebook is the mechanism, by hand, once.

The lesson page for this notebook: `lessons/phase-02-classical-machine-learning/chapter-05-core-supervised-algorithms/lesson-18-knn-svm.html`